In [17]:
from dataclasses import dataclass
from typing import Callable, Any

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    dynamic_prompt as dynamic_prompt_middleware,
    wrap_model_call,
)
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver


@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"


class AuthState(AgentState):
    authenticated: bool = False


@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate a user and update auth state."""
    success = (
        email == runtime.context.email_address
        and password == runtime.context.password
    )
    return Command(
        update={
            "authenticated": success,
            "messages": [
                ToolMessage(
                    content=(
                        "Successfully authenticated"
                        if success
                        else "Authentication failed"
                    ),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


@tool
def check_inbox() -> str:
    """Return recent inbox messages."""
    return "Hi Julie, I'm in town next week. Coffee?"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email reply."""
    return f"Email sent to {to} with subject {subject}"


@wrap_model_call
def dynamic_tool_call(request: Any, handler: Callable):
    """Expose auth tool first, then inbox/send tools after successful auth."""
    authenticated = request.state.get("authenticated", False)
    tools = [check_inbox, send_email] if authenticated else [authenticate]
    return handler(request.override(tools=tools))


@dynamic_prompt_middleware
def auth_prompt(request: Any) -> str:
    """Switch system prompt based on auth state."""
    if request.state.get("authenticated", False):
        return "You are an assistant that can check email and send replies."
    return "You are an assistant that can authenticate users."


agent = create_agent(
    model="gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    state_schema=AuthState,
    context_schema=EmailContext,
    checkpointer=InMemorySaver(),
    middleware=[
        dynamic_tool_call,
        auth_prompt,
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),
    ],
)


In [18]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I found a new message in Julie’s inbox: "Hi Julie, I'm in town next week. Coffee?"

How would you like to respond? Here are a few draft options you can choose from or customize:

Option 1 — Casual and friendly
Hey! Great to hear from you. I’d love to catch up. I’m free Tuesday or Thursday afternoon next week. Do either work for you? There are a few good coffee spots downtown if you want a recommendation.

Option 2 — Clear with proposed times
Hi! I’d love to catch up. Are you free Tuesday morning or Wednesday afternoon next week? If you have a place in mind, let me know the time and location and I’ll be there.

Option 3 — Short and flexible
Sounds good! I’m around next week—tell me a day and a spot that work for you and I’ll make it happen.

If you have a preferred time, location, or tone (more formal, more playful), tell me and I’ll tailor the message. Want me to send one of these now? If yes, which option (or your own draft), and I’ll send it.


In [19]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Draft selected (Option 2):

Subject: Re: Coffee next week
Body:
Hi! I’d love to catch up. Are you free Tuesday morning or Wednesday afternoon next week? If you have a place in mind, let me know the time and location and I’ll be there.

To send this, please provide the recipient's email address (the person who wrote the message in Julie’s inbox). Once you share it, I’ll send it right away. If you’d prefer a different tone or another draft, tell me and I’ll adjust.


In [20]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Draft selected (Option 2):

Subject: Re: Coffee next week
Body:
Hi! I’d love to catch up. Are you free Tuesday morning or Wednesday afternoon next week? If you have a place in mind, let me know the time and location and I’ll be there.

To send this, please provide the recipient's email address (the person who wrote the message in Julie’s inbox). Once you share it, I’ll send it right away. If you’d prefer a different tone or another draft, tell me and I’ll adjust.


In [21]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='59725265-cde1-4d70-9749-819a8ca91d6c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 148, 'total_tokens': 370, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DSDxYcSFwloR8tOL09CKaJgi072dq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6b27-15bf-7c72-b3e3-040a083f3ea6-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_8atCvrWov9qsmmDc7h2WMtV9', 'type': 'tool_call